# HTFRS Fase B: Ingredient Similarity + Community Detection

Implementación de las Ecuaciones 4-7 (ingredient similarity) y simplificación de Eqs. 8-10 (community detection con Louvain).

- **Eqs. 4-5**: tf-isf ingredient weighting
- **Eqs. 6-7**: Food-food similarity
- **Eqs. 8-10**: Community detection (Louvain en vez de LPA) + cluster-based prediction

In [1]:
import numpy as np
import pandas as pd
import os
import sys
import re
import pickle
from collections import defaultdict
from scipy.sparse import csr_matrix, lil_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
import time as time_module

K_RECS = 10
TOP_RECIPES_FOR_GRAPH = 10000  # Recetas para el grafo de clustering
TOP_NEIGHBORS_GRAPH = 10       # Vecinos por nodo en el grafo
TEST_SIZE = 0.2
RANDOM_STATE = 42
MIN_INGREDIENT_FREQ = 5        # Ignorar ingredientes en < N recetas

print('Setup listo')

Setup listo


## 1. Carga de datos

In [2]:
# --- Carga de Reviews ---
try:
    import kagglehub
    path = kagglehub.dataset_download('irkaal/foodcom-recipes-and-reviews')
    reviews = pd.read_csv(os.path.join(path, 'reviews.csv'))
except Exception:
    reviews = pd.read_csv('../Dataset_recetas/reviews.csv')
print(f'Reviews: {len(reviews)}')

# --- Carga de Recipes ---
try:
    import gdown
    url = 'https://drive.google.com/uc?id=1Rl8XowC9N6cxrdiPvH4wToFUvZrrBqRG'
    gdown.download(url, 'recipes_final_consolidado.csv', quiet=True)
    recipes = pd.read_csv('recipes_final_consolidado.csv')
except Exception:
    recipes = pd.read_csv('../Dataset_recetas/recipes_final_consolidado.csv')
print(f'Recipes: {len(recipes)}')

# Filtering
recipes['ExtractedServingSize'] = recipes['ExtractedServingSize'].str.extract(r'\((.*?)\)').astype(float)
recipes = recipes[recipes['ExtractedServingSize'] > 0].copy()
user_counts = reviews['AuthorId'].value_counts()
valid_users = user_counts[user_counts > 1].index
reviews_filtrado = reviews[reviews['AuthorId'].isin(valid_users)].copy()
valid_recipes = set(recipes['RecipeId'])
reviews_filtrado = reviews_filtrado[reviews_filtrado['RecipeId'].isin(valid_recipes)].copy()

# Sellos
recipes['IsHighCalories'] = ((recipes['Calories'] / recipes['ExtractedServingSize']) * 100) >= 275
recipes['IsHighSugar'] = ((recipes['SugarContent'] / recipes['ExtractedServingSize']) * 100) >= 10
recipes['IsHighSaturatedFat'] = ((recipes['SaturatedFatContent'] / recipes['ExtractedServingSize']) * 100) >= 4
recipes['IsHighSodium'] = ((recipes['SodiumContent'] / recipes['ExtractedServingSize']) * 100) >= 400
recipes['num_sellos'] = recipes[['IsHighCalories', 'IsHighSugar', 'IsHighSaturatedFat', 'IsHighSodium']].sum(axis=1).astype(int)
recipe_sellos_dict = dict(zip(recipes['RecipeId'], recipes['num_sellos']))

# Split
train_df, test_df = train_test_split(reviews_filtrado, test_size=TEST_SIZE, random_state=RANDOM_STATE)
print(f'Train: {len(train_df)}, Test: {len(test_df)}')

Using Colab cache for faster access to the 'foodcom-recipes-and-reviews' dataset.
Reviews: 1401982
Recipes: 522359
Train: 959671, Test: 239918


## 2. Parsing de ingredientes (Eqs. 4-5)

In [3]:
def parse_ingredients(text):
    """Parsea el formato c(\"sugar\", \"flour\", ...) a lista de strings."""
    if pd.isna(text):
        return []
    matches = re.findall(r'"([^"]+)"', str(text))
    return [m.lower().strip() for m in matches if len(m.strip()) > 1]

recipes['ingredients_list'] = recipes['RecipeIngredientParts'].apply(parse_ingredients)

# Estadísticas
n_with_ingredients = (recipes['ingredients_list'].str.len() > 0).sum()
all_ingredients = set()
for lst in recipes['ingredients_list']:
    all_ingredients.update(lst)

print(f'Recetas con ingredientes: {n_with_ingredients}/{len(recipes)}')
print(f'Ingredientes únicos: {len(all_ingredients)}')
print(f'Ingredientes promedio por receta: {recipes["ingredients_list"].str.len().mean():.1f}')

Recetas con ingredientes: 519112/520971
Ingredientes únicos: 7295
Ingredientes promedio por receta: 7.9


In [4]:
# Solo recetas con ingredientes
recipes_with_ing = recipes[recipes['ingredients_list'].str.len() > 0].copy()
recipe_ids_ordered = recipes_with_ing['RecipeId'].values
recipeid2idx_ing = {rid: i for i, rid in enumerate(recipe_ids_ordered)}
idx2recipeid_ing = {i: rid for rid, i in recipeid2idx_ing.items()}

# MultiLabelBinarizer -> sparse matrix
mlb = MultiLabelBinarizer(sparse_output=True)
ingredient_matrix = mlb.fit_transform(recipes_with_ing['ingredients_list'])
ingredient_names = mlb.classes_
M = ingredient_matrix.shape[0]

print(f'Matriz de ingredientes: {ingredient_matrix.shape}')
print(f'Densidad: {ingredient_matrix.nnz / (ingredient_matrix.shape[0] * ingredient_matrix.shape[1]) * 100:.2f}%')

Matriz de ingredientes: (519112, 7295)
Densidad: 0.11%


In [5]:
# tf-isf weights (Eqs. 4-5)
# n_j = número de recetas que contienen ingrediente j
n_j = np.array(ingredient_matrix.sum(axis=0)).flatten().astype(float)

# Filtrar ingredientes muy raros o muy comunes
valid_mask = (n_j >= MIN_INGREDIENT_FREQ) & (n_j <= M * 0.5)
print(f'Ingredientes válidos: {valid_mask.sum()} de {len(n_j)}')

# IDF weights
w_j = np.zeros(len(n_j))
w_j[valid_mask] = np.log(M / n_j[valid_mask])

print(f'Peso IDF min (válidos): {w_j[valid_mask].min():.2f}')
print(f'Peso IDF max: {w_j[valid_mask].max():.2f}')
print(f'Peso IDF mean: {w_j[valid_mask].mean():.2f}')

Ingredientes válidos: 4737 de 7295
Peso IDF min (válidos): 1.00
Peso IDF max: 11.55
Peso IDF mean: 8.80


## 3. Food-Food Similarity (Eqs. 6-7)

Solo para las top recetas más populares (para el grafo de clustering).

In [6]:
# Top recetas más populares (por número de ratings en train)
recipe_popularity = train_df['RecipeId'].value_counts()
top_recipe_ids = set(recipe_popularity.head(TOP_RECIPES_FOR_GRAPH).index)

# Filtrar a recetas que tienen ingredientes
top_recipe_ids = top_recipe_ids & set(recipeid2idx_ing.keys())
top_recipe_indices = [recipeid2idx_ing[rid] for rid in top_recipe_ids]

print(f'Top recetas con ingredientes para grafo: {len(top_recipe_indices)}')

Top recetas con ingredientes para grafo: 9958


In [7]:
# Submatriz de ingredientes para top recetas
top_indices = sorted(top_recipe_indices)
sub_ingredient = ingredient_matrix[top_indices]

# Weighted matrix
from scipy.sparse import diags
W_diag = diags(w_j)
weighted_sub = sub_ingredient @ W_diag

# Similarity matrix (Eq. 6-7): weighted dot product
print('Calculando similitud entre recetas...')
start_time = time_module.time()
sim_matrix = (weighted_sub @ sub_ingredient.T).toarray()
elapsed = time_module.time() - start_time
print(f'Similitud calculada en {elapsed:.1f}s')
print(f'Shape: {sim_matrix.shape}')

# Mapeo local
local2global = {i: top_indices[i] for i in range(len(top_indices))}
global2local = {v: k for k, v in local2global.items()}
local2recipeid = {i: idx2recipeid_ing[top_indices[i]] for i in range(len(top_indices))}

Calculando similitud entre recetas...
Similitud calculada en 0.8s
Shape: (9958, 9958)


## 4. Community Detection (Louvain)

Sustituimos el LPA modificado con Laplacian Centrality (Eqs. 8-9) por **Louvain** para eficiencia computacional.

In [8]:
try:
    import networkx as nx
    from community import community_louvain
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'python-louvain', 'networkx'])
    import networkx as nx
    from community import community_louvain

G = nx.Graph()

# Agregar nodos
for i in range(len(top_indices)):
    G.add_node(i)

# Agregar aristas: top-K vecinos por similitud
np.fill_diagonal(sim_matrix, 0)  # No self-loops

for i in range(len(top_indices)):
    top_k_idx = np.argsort(sim_matrix[i])[-TOP_NEIGHBORS_GRAPH:]
    for j in top_k_idx:
        if sim_matrix[i, j] > 0:
            G.add_edge(i, j, weight=float(sim_matrix[i, j]))

print(f'Grafo: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas')

Grafo: 9958 nodos, 83894 aristas


In [9]:
# Louvain community detection
partition = community_louvain.best_partition(G, random_state=RANDOM_STATE)

# Agrupar recetas por cluster
cluster_to_recipes = defaultdict(list)  # {cluster_id: [local_indices]}
recipe_to_cluster = {}  # {recipe_id: cluster_id}

for local_idx, cluster_id in partition.items():
    cluster_to_recipes[cluster_id].append(local_idx)
    rid = local2recipeid[local_idx]
    recipe_to_cluster[rid] = cluster_id

cluster_sizes = [len(v) for v in cluster_to_recipes.values()]
print(f'Clusters: {len(cluster_to_recipes)}')
print(f'Tamaño promedio: {np.mean(cluster_sizes):.1f}')
print(f'Tamaño min/max: {min(cluster_sizes)}/{max(cluster_sizes)}')

Clusters: 15
Tamaño promedio: 663.9
Tamaño min/max: 1/2751


## 5. Cluster-based Prediction (Eq. 10)

$p^{food}_{iu} = \bar{r}_i + \frac{\sum_{j \in C_i} sim(f_i, f_j) \times (r_{ju} - \bar{r}_j)}{\sum_{j \in C_i} |sim(f_i, f_j)|}$

In [10]:
# Estructuras de datos para predicción
# Ratings de train por usuario y por item
user_ratings_train = defaultdict(dict)  # {user: {recipe: rating}}
item_ratings_train = defaultdict(list)  # {recipe: [ratings]}

for row in train_df[['AuthorId', 'RecipeId', 'Rating']].itertuples(index=False):
    user_ratings_train[row[0]][row[1]] = row[2]
    item_ratings_train[row[1]].append(row[2])

item_mean_rating = {it: np.mean(ratings) for it, ratings in item_ratings_train.items()}
global_mean = train_df['Rating'].mean()

# Items de train por usuario
train_items_per_user = defaultdict(set)
for u, items in user_ratings_train.items():
    train_items_per_user[u] = set(items.keys())

print(f'Usuarios con ratings en train: {len(user_ratings_train)}')
print(f'Recetas en clusters: {len(recipe_to_cluster)}')

Usuarios con ratings en train: 70916
Recetas en clusters: 9958


In [11]:
# Lookup rápido recipeid -> local_idx
recipeid2local = {rid: li for li, rid in local2recipeid.items()}

# Pre-computar estructuras por cluster para vectorización
cluster_data = {}
for cid, members in cluster_to_recipes.items():
    members_arr = np.array(members)
    member_rids = [local2recipeid[m] for m in members]
    member_means = np.array([item_mean_rating.get(rid, global_mean) for rid in member_rids])
    # Submatriz de similitud dentro del cluster
    sim_sub = sim_matrix[np.ix_(members_arr, members_arr)]
    cluster_data[cid] = {
        'members': members_arr,
        'rids': member_rids,
        'means': member_means,
        'sim': sim_sub,
        'rid_to_local': {rid: i for i, rid in enumerate(member_rids)},
    }

print(f'Clusters pre-computados: {len(cluster_data)}')

Clusters pre-computados: 15


## 6. Generar recomendaciones food-based

In [12]:
# Pre-computar: clusters relevantes por usuario
user_relevant_clusters = defaultdict(set)
for u, items in user_ratings_train.items():
    for rid in items:
        if rid in recipe_to_cluster:
            user_relevant_clusters[u].add(recipe_to_cluster[rid])

test_users = sorted(set(test_df['AuthorId'].unique()) & set(user_ratings_train.keys()))
print(f'Usuarios de test: {len(test_users)}')
print(f'Usuarios con clusters relevantes: {sum(1 for u in test_users if u in user_relevant_clusters)}')

print('\nGenerando predicciones food-based (vectorizado)...')
start_time = time_module.time()

user_recommendations_food = {}
user_predictions_food = {}

for i, u in enumerate(test_users):
    relevant_clusters = user_relevant_clusters.get(u, set())
    if not relevant_clusters:
        continue

    seen = train_items_per_user.get(u, set())
    user_items = user_ratings_train.get(u, {})
    all_preds = {}

    for cid in relevant_clusters:
        cd = cluster_data[cid]
        rids = cd['rids']
        means = cd['means']
        sim_sub = cd['sim']
        n = len(rids)

        # Máscara: qué miembros del cluster rateó este usuario
        rated_mask = np.array([rid in user_items for rid in rids])
        if not rated_mask.any():
            continue

        # Desviaciones del usuario respecto al promedio del ítem
        deviations = np.zeros(n)
        for j in np.where(rated_mask)[0]:
            deviations[j] = user_items[rids[j]] - means[j]

        # Candidatos: no vistos por el usuario
        candidate_mask = np.array([rid not in seen for rid in rids])
        candidate_indices = np.where(candidate_mask)[0]

        if len(candidate_indices) == 0:
            continue

        # Vectorizado: para todos los candidatos a la vez
        # sim_sub[candidates, :][:, rated] @ deviations[rated]
        sim_to_rated = sim_sub[np.ix_(candidate_indices, np.where(rated_mask)[0])]
        devs_rated = deviations[rated_mask]

        numerators = sim_to_rated @ devs_rated
        denominators = np.abs(sim_to_rated).sum(axis=1)

        for idx, ci in enumerate(candidate_indices):
            if denominators[idx] == 0:
                pred = means[ci]
            else:
                pred = means[ci] + numerators[idx] / denominators[idx]
            pred = np.clip(pred, 0, 5)
            all_preds[rids[ci]] = float(pred)

    if all_preds:
        sorted_items = sorted(all_preds.items(), key=lambda x: x[1], reverse=True)
        user_recommendations_food[u] = [item_id for item_id, _ in sorted_items[:K_RECS]]
        user_predictions_food[u] = all_preds

    if (i + 1) % 100 == 0:
        elapsed = time_module.time() - start_time
        rate = (i + 1) / elapsed
        remaining = (len(test_users) - i - 1) / rate
        print(f'  {i+1}/{len(test_users)} ({rate:.1f} users/s, ~{remaining/60:.0f} min restantes)')

elapsed = time_module.time() - start_time
print(f'\nPredicciones generadas en {elapsed/60:.1f} min')
print(f'Usuarios con recomendaciones: {len(user_recommendations_food)}')

Usuarios de test: 41698
Usuarios con clusters relevantes: 32949

Generando predicciones food-based (vectorizado)...
  100/41698 (60.5 users/s, ~11 min restantes)
  200/41698 (57.6 users/s, ~12 min restantes)
  300/41698 (58.2 users/s, ~12 min restantes)
  400/41698 (57.0 users/s, ~12 min restantes)
  500/41698 (57.6 users/s, ~12 min restantes)
  600/41698 (57.1 users/s, ~12 min restantes)
  700/41698 (57.0 users/s, ~12 min restantes)
  800/41698 (56.6 users/s, ~12 min restantes)
  900/41698 (55.1 users/s, ~12 min restantes)
  1000/41698 (55.4 users/s, ~12 min restantes)
  1100/41698 (55.8 users/s, ~12 min restantes)
  1200/41698 (55.9 users/s, ~12 min restantes)
  1300/41698 (55.9 users/s, ~12 min restantes)
  1400/41698 (55.4 users/s, ~12 min restantes)
  1500/41698 (54.9 users/s, ~12 min restantes)
  1600/41698 (55.1 users/s, ~12 min restantes)
  1700/41698 (54.7 users/s, ~12 min restantes)
  1800/41698 (54.5 users/s, ~12 min restantes)
  1900/41698 (54.8 users/s, ~12 min restantes)


## 7. Evaluación

In [13]:
# === evaluation_utils (inline para Colab) ===
import numpy as np

RELEVANCE_THRESHOLD = 4

def get_relevant_items(test_df, user_id, threshold=RELEVANCE_THRESHOLD):
    user_test = test_df[test_df['AuthorId'] == user_id]
    return set(user_test[user_test['Rating'] >= threshold]['RecipeId'])

def precision_at_k(recommended, relevant, k=10):
    return len(set(recommended[:k]) & set(relevant)) / k

def recall_at_k(recommended, relevant, k=10):
    rel_set = set(relevant)
    if len(rel_set) == 0:
        return 0.0
    return len(set(recommended[:k]) & rel_set) / len(rel_set)

def f1_at_k(recommended, relevant, k=10):
    p = precision_at_k(recommended, relevant, k)
    r = recall_at_k(recommended, relevant, k)
    if p + r == 0:
        return 0.0
    return 2 * p * r / (p + r)

def _dcg_at_k(relevances, k):
    relevances = np.asarray(relevances)[:k]
    if relevances.size:
        return np.sum((2**relevances - 1) / np.log2(np.arange(2, relevances.size + 2)))
    return 0.0

def ndcg_at_k(recommended, relevant, k=10):
    relevances = [1 if r in relevant else 0 for r in recommended[:k]]
    dcg = _dcg_at_k(relevances, k)
    ideal = _dcg_at_k(sorted(relevances, reverse=True), k)
    return dcg / ideal if ideal > 0 else 0.0

def average_precision_at_k(recommended, relevant, k=10):
    hits = 0
    sum_prec = 0.0
    rel_set = set(relevant)
    for i, item in enumerate(recommended[:k]):
        if item in rel_set:
            hits += 1
            sum_prec += hits / (i + 1)
    return sum_prec / min(len(rel_set), k) if rel_set else 0.0

def sellos_at_k(recommended, recipe_sellos_dict, k=10):
    sellos = [recipe_sellos_dict.get(r, 0) for r in recommended[:k]]
    return np.mean(sellos) if sellos else 0.0

def sello_free_at_k(recommended, recipe_sellos_dict, k=10):
    sellos = [recipe_sellos_dict.get(r, 0) for r in recommended[:k]]
    return np.mean([1 if s == 0 else 0 for s in sellos]) if sellos else 0.0

def evaluate_recommendations(recommended, relevant, recipe_sellos_dict, k=10):
    return {
        'P@K': precision_at_k(recommended, relevant, k),
        'R@K': recall_at_k(recommended, relevant, k),
        'F1@K': f1_at_k(recommended, relevant, k),
        'nDCG@K': ndcg_at_k(recommended, relevant, k),
        'MAP@K': average_precision_at_k(recommended, relevant, k),
        'S@K': sellos_at_k(recommended, recipe_sellos_dict, k),
        'SS@K': sello_free_at_k(recommended, recipe_sellos_dict, k),
    }

def evaluate_all_users(user_recommendations, test_df, recipe_sellos_dict,
                       k=10, threshold=RELEVANCE_THRESHOLD):
    all_metrics = []
    for user_id, recommended in user_recommendations.items():
        relevant = get_relevant_items(test_df, user_id, threshold)
        metrics = evaluate_recommendations(recommended, relevant, recipe_sellos_dict, k)
        all_metrics.append(metrics)
    if not all_metrics:
        return {}
    return {key: np.mean([m[key] for m in all_metrics]) for key in all_metrics[0]}

# --- Evaluar ---
metrics_food = evaluate_all_users(
    user_recommendations_food, test_df, recipe_sellos_dict, k=K_RECS
)

print('\n=== Resultados Food-Based (Eqs. 4-10, Louvain) ===')
print(f'Usuarios evaluados: {len(user_recommendations_food)}')
print(f'K = {K_RECS}')
print(f'Clusters: {len(cluster_to_recipes)}')
print()
for metric, value in metrics_food.items():
    print(f'  {metric}: {value:.6f}')


=== Resultados Food-Based (Eqs. 4-10, Louvain) ===
Usuarios evaluados: 32947
K = 10
Clusters: 15

  P@K: 0.000222
  R@K: 0.000729
  F1@K: 0.000247
  nDCG@K: 0.000978
  MAP@K: 0.000292
  S@K: 0.878107
  SS@K: 0.546092


## 8. Guardar resultados

In [14]:
output = {
    'user_recommendations_food': user_recommendations_food,
    'user_predictions_food': user_predictions_food,
    'recipe_to_cluster': recipe_to_cluster,
    'cluster_to_recipes': dict(cluster_to_recipes),
    'item_mean_rating': item_mean_rating,
    'metrics_food': metrics_food,
    'params': {
        'top_recipes_for_graph': TOP_RECIPES_FOR_GRAPH,
        'top_neighbors_graph': TOP_NEIGHBORS_GRAPH,
        'min_ingredient_freq': MIN_INGREDIENT_FREQ,
        'n_clusters': len(cluster_to_recipes),
    }
}

import pickle
with open('htfrs_food_output.pkl', 'wb') as f:
    pickle.dump(output, f)

print('Resultados guardados en htfrs_food_output.pkl')
print(f'Tamaño: {os.path.getsize("htfrs_food_output.pkl") / 1e6:.1f} MB')

Resultados guardados en htfrs_food_output.pkl
Tamaño: 2023.1 MB
